In [1]:
import numpy as np
import time
from numba import njit
import jax.numpy as jnp
from jax import jit as jax_jit

# Define Numba functions
@njit
def numba_operations(x):
    norm_x = x / x.sum()
    exp_x = np.exp(norm_x)
    dot_x = np.dot(exp_x, exp_x.T)
    return dot_x

# Define JAX functions
@jax_jit
def jax_operations(x):
    norm_x = x / x.sum()
    exp_x = jnp.exp(norm_x)
    dot_x = jnp.dot(exp_x, exp_x.T)
    return dot_x

# Generate random 40x40 array
x_np = np.random.rand(40, 40).astype(np.float32)
x_jax = jnp.array(x_np)

# Warm-up Numba
for _ in range(10): numba_operations(x_np)
# Benchmark Numba
start_time = time.time()
for _ in range(1000): numba_operations(x_np)
numba_time = time.time() - start_time

# Warm-up JAX
for _ in range(10): jax_operations(x_jax).block_until_ready()
# Benchmark JAX
start_time = time.time()
for _ in range(1000): jax_operations(x_jax).block_until_ready()
jax_time = time.time() - start_time

print(f"Numba Time: {numba_time:.6f} seconds")
print(f"JAX Time: {jax_time:.6f} seconds")


No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


Numba Time: 0.007723 seconds
JAX Time: 0.016514 seconds


In [1]:
import numpy as np
from numba import njit
from numba.core import types
from numba.typed import Dict

# Make array type.  Type-expression is not supported in jit
# functions.
float_array = types.float64[:]

@njit
def foo():
    # Make dictionary
    d = Dict.empty(
        key_type=types.unicode_type,
        value_type=float_array,
    )
    # Fill the dictionary
    d["posx"] = np.arange(3).astype(np.float64)
    d["posy"] = np.eye(3, 6).astype(np.float64)
    return d

d = foo()
# Print the dictionary
print(d)  # Out: {posx: [0. 1. 2.], posy: [3. 4. 5.]}

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
[1m[1mNo implementation of function Function(<built-in function setitem>) found for signature:
 
 >>> setitem(DictType[unicode_type,array(float64, 1d, A)]<iv=None>, Literal[str](posy), array(float64, 2d, C))
 
There are 16 candidate implementations:
[1m  - Of which 14 did not match due to:
  Overload of function 'setitem': File: <numerous>: Line N/A.
    With argument(s): '(DictType[unicode_type,array(float64, 1d, A)]<iv=None>, unicode_type, array(float64, 2d, C))':[0m
[1m   No match.[0m
[1m  - Of which 2 did not match due to:
  Overload in function 'impl_setitem': File: numba\typed\dictobject.py: Line 706.
    With argument(s): '(DictType[unicode_type,array(float64, 1d, A)]<iv=None>, unicode_type, array(float64, 2d, C))':[0m
[1m   Rejected as the implementation raised a specific error:
     LoweringError: Failed in nopython mode pipeline (step: native lowering)
   [1m[1mexpecting {i8*, i8*, i64, i64, do

In [2]:
types.NamedTuple

numba.core.types.containers.NamedTuple

In [1]:
from numba import jit
from numba.typed import List
from numba import types
import numpy as np
from collections import namedtuple

# Define the Python named tuple and the corresponding Numba type
MyTuple = namedtuple('MyTuple', ['a', 'b', 'c'])
MyTupleType = types.NamedTuple([types.int32, types.float64, types.boolean], MyTuple)

@jit(nopython=True)
def create_and_use_tuples():
    t = MyTuple(np.int32(1), 2.0, True)  # Explicit type casting for int32
    result_a = t.a * 2
    result_b = t.b / 2
    result_c = not t.c
    tuple_list = List.empty_list(MyTupleType)
    tuple_list.append(t)
    tuple_list.append(MyTuple(np.int32(result_a), result_b, result_c))
    return tuple_list

# Run the function
create_and_use_tuples()


ListType[MyTuple(int32, float64, bool)]([MyTuple(a=1, b=2.0, c=True), MyTuple(a=2, b=1.0, c=False), ...])